In [1]:
# Step 1: Download the file
url = "https://neural-nexus-2.s3.ap-south-1.amazonaws.com/Education.zip"
output_path = "/content/Education.zip"

!wget -O {output_path} {url}

# Step 2: Extract the ZIP file
import zipfile
import os

extract_path = "/content/Education"

# Create directory if not exists
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(output_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Download and extraction complete!")

--2026-04-15 08:32:27--  https://neural-nexus-2.s3.ap-south-1.amazonaws.com/Education.zip
Resolving neural-nexus-2.s3.ap-south-1.amazonaws.com (neural-nexus-2.s3.ap-south-1.amazonaws.com)... 16.12.36.114, 3.5.209.106, 3.5.209.72, ...
Connecting to neural-nexus-2.s3.ap-south-1.amazonaws.com (neural-nexus-2.s3.ap-south-1.amazonaws.com)|16.12.36.114|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20185858205 (19G) [application/zip]
Saving to: ‘/content/Education.zip’

/content/Education. 100%[===================>]  18.80G  16.7MB/s    in 22m 42s 

2026-04-15 08:55:10 (14.1 MB/s) - ‘/content/Education.zip’ saved [20185858205/20185858205]

✅ Download and extraction complete!


In [2]:
!pip install opencv-python torch torchvision tqdm

In [9]:
import os
import cv2
import torch
from torch.utils.data import Dataset
import numpy as np

class VideoDataset(Dataset):
    def __init__(self, root_dir, seq_len=8):
        self.samples = []
        self.seq_len = seq_len

        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for video in os.listdir(cls_path):
                self.samples.append((os.path.join(cls_path, video), self.class_to_idx[cls]))

    def __len__(self):
        return len(self.samples)

    def load_video(self, path):
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        if total <= 0:
            cap.release()
            return None

        idxs = np.linspace(0, total-1, self.seq_len).astype(int)

        frames = []
        for i in idxs:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if not ret:
                continue
            frame = cv2.resize(frame, (112,112))
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            return None

        while len(frames) < self.seq_len:
            frames.append(frames[-1])

        return np.stack(frames)

    def __getitem__(self, idx):
        while True:
            path, label = self.samples[idx]
            frames = self.load_video(path)

            if frames is not None:
                frames = torch.from_numpy(frames).permute(0,3,1,2)  # (T,C,H,W)
                return frames, label

            idx = np.random.randint(0, len(self.samples))

In [13]:
import torch.nn as nn
import torch
import torchvision.models as models
from torchvision.models import ResNet18_Weights

# 🔥 Attention pooling
class AttentionPooling(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        out = (weights * x).sum(dim=1)
        return out, weights


class HybridModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # CNN backbone
        self.cnn = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        self.cnn.layer4 = nn.Identity()   # 🔥 remove last heavy block
        self.cnn.fc = nn.Identity()
        self.feature_dim = 256

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.feature_dim,
            nhead=4,   # 🔥 reduced
            dim_feedforward=512,  # 🔥 smaller
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)  # 🔥 reduced
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        # Attention pooling
        self.attn_pool = AttentionPooling(self.feature_dim)

        self.fc = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape

        # normalize on GPU
        x = x.float() / 255.0

        # 🔥 CNN features
        x = x.view(B*T, C, H, W)
        feat = self.cnn(x)
        feat = feat.view(B, T, -1)

        # 🔥 MOTION (frame difference)
        motion = feat[:, 1:, :] - feat[:, :-1, :]
        motion = torch.cat([motion, motion[:, -1:, :]], dim=1)

        # combine
        feat = feat + motion

        # 🔥 Transformer
        feat = self.transformer(feat)

        # 🔥 Attention pooling
        context, weights = self.attn_pool(feat)

        out = self.fc(context)

        return out, weights

In [ ]:
from torch.utils.data import DataLoader, random_split
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = VideoDataset("/content/Education/Education")

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(
    train_ds,
    batch_size=24,
    shuffle=True,
    num_workers=3,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=24,
    shuffle=False,
    num_workers=3,
    persistent_workers=True
)
model = HybridModel(len(dataset.classes)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

scaler = torch.amp.GradScaler(device="cuda")

train_losses = []
val_accs = []

for epoch in range(8):
    model.train()
    total_loss = 0

    for videos, labels in tqdm(train_loader):
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs, _ = model(videos)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    train_losses.append(total_loss / len(train_loader))

    # VALIDATION
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos = videos.to(device)
            labels = labels.to(device)

            outputs, _ = model(videos)
            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    acc = correct / total
    val_accs.append(acc)

    print(f"Epoch {epoch} | Loss {train_losses[-1]:.4f} | Val Acc {acc:.4f}")

100%|██████████| 2404/2404 [10:50<00:00,  3.70it/s]


Epoch 0 | Loss 1.8774 | Val Acc 0.3559


100%|██████████| 2404/2404 [10:56<00:00,  3.66it/s]


Epoch 1 | Loss 1.5057 | Val Acc 0.3785


100%|██████████| 2404/2404 [10:55<00:00,  3.67it/s]


Epoch 2 | Loss 1.3247 | Val Acc 0.4016


 16%|█▌        | 390/2404 [01:46<08:21,  4.02it/s]

In [ ]:
plt.plot(train_losses, label="Loss")
plt.plot(val_accuracies, label="Val Accuracy")
plt.legend()
plt.show()

In [ ]:
model.eval()

videos, _ = next(iter(val_loader))
videos = videos.to(device)

_, weights = model(videos)

weights = weights[0].detach().cpu().numpy()

plt.imshow(weights.T, cmap='hot', aspect='auto')
plt.title("Attention Heatmap")
plt.xlabel("Frame")
plt.ylabel("Importance")
plt.colorbar()
plt.show()